In [ ]:
import numpy as np

from utils import  load_cylinder_dataset
import os

X_true, X_noisy, simulation_dir = load_cylinder_dataset(noise_type='gauss', 
                                                        noise_level=0.02, 
                                                        smoothing=0.01,
                                                        visualize=True,
                                                        )

Separate the dataset into data to train the POD-ESN and data to test the POD-ESN-EnKF

In [ ]:

N_train = 200


X_train, X_train_true = [yy[:N_train].T for yy in [X_noisy, X_true]]
X_filter, X_filter_true = [yy[N_train:].T for yy in [X_noisy, X_true]]



dt = 0.01
t_val = X_train.shape[-1] *.1 * dt 
t_train = X_train.shape[-1] * dt - t_val




# Part I. State estimation


In [ ]:
from models_data_driven import POD_ESN


case_POD_ESN = POD_ESN(data=X_train,
                       
                    # ====== POD arguments ======== # 
                    domain=[-2.5, 2.5, 0, 12],  
                    N_modes=4,  
                    qr_selection = True,
                    N_q=2,

                    # ====== ESN arguments ======== # 
                    t_val=t_val,
                    t_train=t_train,
                    t_test=0.,
                    N_units=40, 
                    train_ESN=True,
                    N_wash=5,
                    noise=1E-4,
                    N_func_evals=26,
                    rho_range=[0.2, 0.9],
                    upsample=2,            # NB: I think there is a bug with upsample > 1. I am working on this
                    Nq=2, 
                  
                    # ====== Model arguments ======== # 
                    dt = dt,
                    figs_folder=simulation_dir, 
                    t_CR=0.5,
                    plot_case=True
                    )


In [ ]:
from utils import set_cylinder_truth
from plot_results import plot_truth

truth = set_cylinder_truth(case_POD_ESN, 
                            X_filter=X_filter, 
                            X_filter_true=X_filter_true, 
                            Nt_obs=10, 
                            visualize=True
                            )

plot_truth(**truth)

Define the ensemble 

In [ ]:
from create import create_ensemble


ensemble = create_ensemble(model=case_POD_ESN,
                           m=10,
                           std_psi=1.1)



In [ ]:
from post_processing.cylinder import view_ensemble

view_ensemble(ensemble)


In [ ]:
from post_processing.cylinder import plot_initial_case

# This saves a file with plots of the initial case and ensemble (including the fig above)
plot_initial_case(ensemble, X_train, X_train_true,
                  figs_folder=simulation_dir, name='initial_case_state-only') 

In [ ]:

from data_assimilation import dataAssimilation


# Perform assimilation
filter_ens = dataAssimilation(ensemble=ensemble,
                              std_obs=0.15,
                              y_obs=truth['y_obs'],
                              t_obs=truth['t_obs'],
                              Nt_extra=truth['Nt_extra']
                              )

In [ ]:


# from post_processing.cylinder import plot_timeseries
from plot_results import plot_timeseries

plot_timeseries(filter_ens, 
                truth, 
                plot_ensemble_members=1)


# Part II. State & Parameter estimation: online learning of Wout
Why us the SVD interesting in this scenario? Because we can use data assimialtion to modify on the fly the value of the singluar values. This is, we can perform state and parameter inference with the ESN on the fly. In machine learning jargon, this is known as *online learning*.

With this objective, we first need to create an ensemble of POD-ESN with different SVDs for each ensemble member. This is achieved by providing `Wout` as the parameter to estimate, i.e., as the `est_a` argument. The `sta_a` parameter indicates the uncewrtainty around the Wout SVDs.


## Singular value decomposition of Wout
We can decompose the trained Wout into its singular value components using the library ```scipy.linalg```. The multiplication of the three matrices are equivalent to the original Wout. 

In [ ]:
import scipy.linalg as sla
import matplotlib.pyplot as plt
# Compute the SVD of the output matrix
[Wout_U, Wout_eigs, Wout_Vh] = sla.svd(ensemble.Wout, full_matrices=False)


ensemble.plot_Wout()


# # #display the three matrices
# fig, axs = plt.subplots(1, 4, figsize=(15, 15), width_ratios=[1, 1, 1, 1])
# for W, ax, title in zip([ensemble.Wout, Wout_U, np.diag(Wout_eigs), Wout_Vh], axs, 
#                         ['$\\mathbf{W_{out}} = $', '$\\mathbf{U}$', '$\\Sigma$', '$\\mathbf{V}^\\mathrm{T}$']):
#     im = ax.imshow(W, cmap='PuOr', vmin=-np.max(W), vmax=np.max(W))
#     ax.set(title=title)
#     # set the same colorbar for all the matrices
#     fig.colorbar(im, ax=ax, shrink=.9, orientation='horizontal')

print('Dimensions of matrices:', Wout_U.shape, Wout_eigs.shape, Wout_Vh.shape)

assert np.allclose(ensemble.Wout, np.dot(Wout_U, np.dot(np.diag(Wout_eigs), Wout_Vh)))

In [ ]:
# Define ensemble
ensemble_2 = create_ensemble(model=case_POD_ESN,
                            filter='EnSRKF', 
                            model_bias=None,
                            m=20,
                            Nt_transient = 500, 
                            std_psi=0.1,      
                            est_a=['Wout'], # Estimate the Wout singular values
                            std_a=0.1       # Standard deviation of the noise added to the Wout singular values
                            )



### Wout as SVD
Now the output matrix is decomposed in 3 matrices. the diagonal of the central one is the matrix of singular valuess. these we can modify in time via real-time DA

In [ ]:

ensemble_2.plot_Wout()  

### Visualize the time evolution of the physical states. 
Uncertainty in the parameters is also shown as increased uncertainty in the states


In [ ]:


view_ensemble(ensemble_2)

In [ ]:

# Perform assimilation
filter_ens = dataAssimilation(ensemble=ensemble_2,
                              std_obs=0.01,
                              y_obs=truth['y_obs'],
                              t_obs=truth['t_obs'],
                              Nt_extra=truth['Nt_extra']
                              )

In [ ]:
from plot_results import plot_parameters, plot_timeseries

plot_timeseries(filter_ens, truth, plot_ensemble_members=True,dims=[0,1])
plot_parameters(filter_ens, truth)
